# Bootstrap the VisDrone benchmark from GitHub

Run this notebook directly from GitHub in a fresh Colab session. It clones or safely updates the disposable checkout, mounts Drive, installs only the shared stack, and runs read-only diagnostics. It never starts model training.

In [ ]:
REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_PATH = "/content/aerial-object-detection-benchmark"
REFERENCE_TYPE = "branch"  # branch, tag, or commit
REFERENCE = "main"
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
MOUNT_GOOGLE_DRIVE = True
INSTALL_SHARED_DEPENDENCIES = True

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if not IN_COLAB:
    raise RuntimeError("Open this bootstrap notebook in Google Colab.")
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

repository = Path(REPOSITORY_PATH)
if not repository.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(repository)], check=True)
elif not (repository / ".git").exists():
    raise RuntimeError(f"Existing path is not a Git checkout: {repository}")
sys.path.insert(0, str(repository))

from src.colab_setup import checkout_repository_ref
state = checkout_repository_ref(repository, REFERENCE, REFERENCE_TYPE)
print(state)

In [ ]:
if INSTALL_SHARED_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repository / "requirements/legacy-colab.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repository), "--no-deps"], check=True)

from src.colab_setup import initialize_drive_directories, validate_drive_writable
from scripts.diagnostics.run_diagnostics import build_report

paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(paths.root)
report = build_report(repository)
print(report)
if report["repository"]["dirty"]:
    raise RuntimeError("Bootstrap must finish with a clean Git checkout.")

Bootstrap complete. Continue with `00_prepare_visdrone.ipynb` at the same selected commit. Large artifacts remain under `DRIVE_ROOT`.